In [3]:
# ============================================================================
# EXPERIMENT 4B — FP16 BASELINE THROUGH THE UNIFIED HARNESS
# cloud notebook Notebook — GPU: T4 x2  |  Internet: ON  |  FRESH SESSION
#
# WHY: Base INT4 scored 98% on Custom-50 in the new harness vs 92% in the
# old Experiment 2 harness. To make the paper's numbers consistent, ALL
# configs must run through this exact harness. This notebook runs FP16
# (the last missing config) and then builds the FINAL combined tables:
#
#   TABLE 4 (final): Custom-50   — FP16 vs INT4 vs INT4+QLoRA
#   TABLE 5 (final): GSM8K + HotpotQA — FP16 vs INT4 vs INT4+QLoRA
#
# The INT4 and INT4+QLoRA numbers from your Experiment 4 run are already
# filled into CELL 12 below, so this notebook is fully self-contained.
#
# ESTIMATED RUNTIME: ~1.5 - 2.5 hours (one config, 250 tasks).
# FP16 Phi-3-mini is ~7.6 GB and fits on one T4 (15 GB). No bitsandbytes,
# no PEFT, no training in this notebook — inference only.
# ============================================================================


# %% ==========================================================================
# CELL 1 — INSTALL (then Run > Restart & Clear Cell Outputs)
# Note: NO bitsandbytes needed — FP16 is unquantized.
# ==============================================================================
# """
# !pip install -q -U "transformers==4.41.2" "accelerate==0.30.1"
# !pip install -q "datasets==2.19.1" wandb
# print("Install complete. NOW RESTART THE KERNEL before continuing.")
# """


# %% ==========================================================================
# CELL 2 — RECORD ENVIRONMENT VERSIONS (for the paper's reproducibility
# statement). Also run  !pip show bitsandbytes  in your INT4 session and
# write the version down — you will pin it in the paper.
# ==============================================================================
# """
# import transformers, torch, datasets
# print("transformers:", transformers.__version__)
# print("torch:", torch.__version__)
# print("datasets:", datasets.__version__)
# print("cuda:", torch.version.cuda)
# print("gpu:", torch.cuda.get_device_name(0))
# """


# %% ==========================================================================
# CELL 3 — IMPORTS + LOAD MODEL IN FP16 (no quantization)
# ==============================================================================
import os, re, gc, json, time, string
import torch
import pandas as pd
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    StoppingCriteria, StoppingCriteriaList,
)

torch.manual_seed(42)

MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map={"": 0},               # FP16, single GPU — NO quantization_config
    attn_implementation="eager",
    torch_dtype=torch.float16,
)
model.config.use_cache = True
model.eval()
print(f"FP16 model loaded. Footprint: {model.get_memory_footprint()/1e9:.2f} GB")


# %% ==========================================================================
# CELL 4 — AGENT ENGINE (identical to Experiment 4, byte for byte)
# ==============================================================================

class StopOnObservation(StoppingCriteria):
    def __init__(self, tokenizer, prompt_len, min_new_tokens=20):
        self.tokenizer = tokenizer
        self.prompt_len = prompt_len
        self.min_new_tokens = min_new_tokens
    def __call__(self, input_ids, scores, **kwargs):
        if input_ids.shape[1] - self.prompt_len < self.min_new_tokens:
            return False
        text = self.tokenizer.decode(
            input_ids[0][self.prompt_len:], skip_special_tokens=True)
        return text.rstrip().endswith("Observation:")


def trim_drift(text):
    for marker in ["---", "**Solution", "**Question", "<|user|>", "<|end|>",
                   "Task:", "\nQuestion:"]:
        idx = text.find(marker)
        if idx > 0:
            text = text[:idx]
    return text


def run_agent(task, tools, system_prompt, max_steps=6, max_new_tokens=220):
    prompt = (f"<|system|>\n{system_prompt}<|end|>\n"
              f"<|user|>\nTask: {task}<|end|>\n<|assistant|>\n")
    trace, tool_calls = "", 0
    t0 = time.time()

    for _ in range(max_steps):
        inputs = tokenizer(prompt + trace, return_tensors="pt",
                           truncation=True, max_length=3600).to(model.device)
        plen = inputs.input_ids.shape[1]
        stopping = StoppingCriteriaList([StopOnObservation(tokenizer, plen)])
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=False,
                temperature=None, top_p=None, stopping_criteria=stopping,
                pad_token_id=tokenizer.eos_token_id,
            )
        trace += trim_drift(
            tokenizer.decode(out[0][plen:], skip_special_tokens=True))

        if "Final Answer:" in trace or "\nAnswer:" in trace:
            break

        if trace.rstrip().endswith("Observation:"):
            actions = re.findall(r"Action:\s*(\w+)\[(.*?)\]", trace, re.DOTALL)
            if actions:
                name, arg = actions[-1]
                name = name.lower().strip()
                if name in tools:
                    result = tools[name](arg)
                    tool_calls += 1
                    trace = trace.rstrip() + f" {result}\n"
                else:
                    trace = trace.rstrip() + " Error: unknown tool.\n"
            else:
                trace = trace.rstrip() + " Error: no valid Action found.\n"
        else:
            break

    latency = time.time() - t0
    m = re.search(r"Final Answer:\s*(.*?)(?:\n|$)", trace, re.DOTALL)
    if not m:
        m = re.search(r"\nAnswer:\s*(.*?)(?:\n|$)", trace, re.DOTALL)
    final = m.group(1).strip() if m else trace.strip().split("\n")[-1]
    steps = len(re.findall(r"Action:", trace))
    return final, steps, tool_calls, latency, trace


def tool_calculator(expr):
    try:
        expr = expr.replace("^", "**").replace(",", "").replace("$", "")
        expr = expr.replace("%", "/100")
        if not re.fullmatch(r"[0-9+\-*/().\s]+", expr):
            return "Error: invalid expression."
        result = eval(expr, {"__builtins__": {}}, {})
        if isinstance(result, float) and result.is_integer():
            result = int(result)
        return str(round(result, 6) if isinstance(result, float) else result)
    except Exception as e:
        return f"Error: {e}"

print("Agent engine ready.")


# %% ==========================================================================
# CELL 5 — BENCHMARK A: CUSTOM 50-TASK BENCHMARK (identical to Experiment 4)
# ==============================================================================

KNOWLEDGE_BASE = {
    "population of paris": "The population of Paris is 2.1 million.",
    "population of tokyo": "The population of Tokyo is 14 million.",
    "population of new delhi": "The population of New Delhi is 32 million.",
    "population of delhi": "The population of New Delhi is 32 million.",
    "capital of japan": "The capital of Japan is Tokyo.",
    "capital of india": "The capital of India is New Delhi.",
    "capital of france": "The capital of France is Paris.",
    "ceo of microsoft": "The CEO of Microsoft is Satya Nadella.",
    "phi-3": "Phi-3-mini has 3.8 billion parameters.",
    "parameters": "Phi-3-mini has 3.8 billion parameters.",
    "speed of light": "The speed of light is 299792 km/s.",
    "tallest mountain": "The tallest mountain is Mount Everest at 8849 meters.",
    "mount everest": "The tallest mountain is Mount Everest at 8849 meters.",
    "longest river": "The longest river is the Nile at 6650 km.",
    "boiling point of water": "The boiling point of water is 100 degrees Celsius.",
    "capital of canada": "The capital of Canada is Ottawa.",
    "population of ottawa": "The population of Ottawa is 1.4 million.",
    "ceo of apple": "The CEO of Apple is Tim Cook.",
    "largest desert": "The largest desert is the Sahara at 9.2 million square km.",
    "population of sydney": "The population of Sydney is 5.3 million.",
    "population of cairo": "The population of Cairo is 22 million.",
    "continents": "There are 7 continents on Earth.",
    "distance from earth to the moon": "The distance from Earth to the Moon is 384400 km.",
    "earth to the moon": "The distance from Earth to the Moon is 384400 km.",
    "body temperature": "The normal human body temperature is 37 degrees Celsius.",
    "capital of brazil": "The capital of Brazil is Brasilia.",
    "largest ocean": "The largest ocean is the Pacific Ocean.",
}

def tool_search(query):
    q = query.lower().strip()
    for key, val in KNOWLEDGE_BASE.items():
        if key in q:
            return val
    best, best_score = None, 0
    for key, val in KNOWLEDGE_BASE.items():
        score = len(set(key.split()) & set(q.split()))
        if score > best_score:
            best, best_score = val, score
    return best if best else "No results found."

CUSTOM_SYSTEM_PROMPT = """You are a helpful AI agent. You solve tasks step by step using tools.

Available tools:
- search[query]: searches for factual information
- calculator[expression]: evaluates a math expression

Use this exact format:
Thought: <your reasoning>
Action: <tool>[<input>]
Observation: <tool result>
... (repeat Thought/Action/Observation as needed)
Thought: I now know the answer.
Final Answer: <the answer>"""

CUSTOM_TOOLS = {"search": tool_search, "calculator": tool_calculator}

BENCHMARK_50 = [
    {"id": 1,  "category": "single_lookup", "task": "What is the population of Paris?", "expected": "2.1"},
    {"id": 2,  "category": "single_lookup", "task": "What is the capital of Japan?", "expected": "tokyo"},
    {"id": 3,  "category": "single_lookup", "task": "Who is the CEO of Microsoft?", "expected": "nadella"},
    {"id": 4,  "category": "single_lookup", "task": "What is the tallest mountain?", "expected": "everest"},
    {"id": 5,  "category": "single_lookup", "task": "What is the boiling point of water?", "expected": "100"},
    {"id": 6,  "category": "arithmetic", "task": "What is 340 multiplied by 25?", "expected": "8500"},
    {"id": 7,  "category": "arithmetic", "task": "What is 15 percent of 8000?", "expected": "1200"},
    {"id": 8,  "category": "arithmetic", "task": "What is 999 plus 111?", "expected": "1110"},
    {"id": 9,  "category": "arithmetic", "task": "What is the square of 47?", "expected": "2209"},
    {"id": 10, "category": "arithmetic", "task": "What is 7200 divided by 8?", "expected": "900"},
    {"id": 11, "category": "multi_step", "task": "What is double the population of Paris in millions?", "expected": "4.2"},
    {"id": 12, "category": "multi_step", "task": "Find the population of Tokyo and add 5 million to it.", "expected": "19"},
    {"id": 13, "category": "multi_step", "task": "How many parameters does Phi-3-mini have, multiplied by 2?", "expected": "7.6"},
    {"id": 14, "category": "multi_step", "task": "What is the population of Paris plus the population of Tokyo, in millions?", "expected": "16.1"},
    {"id": 15, "category": "multi_step", "task": "Search for the speed of light in km/s and divide it by 1000.", "expected": "299.79"},
    {"id": 16, "category": "tool_selection", "task": "What is 456 plus 544? Verify with a tool.", "expected": "1000"},
    {"id": 17, "category": "tool_selection", "task": "What is the capital of France?", "expected": "paris"},
    {"id": 18, "category": "tool_selection", "task": "Compute 12 times 12.", "expected": "144"},
    {"id": 19, "category": "tool_selection", "task": "What is the longest river?", "expected": "nile"},
    {"id": 20, "category": "tool_selection", "task": "What is 25 percent of 400?", "expected": "100"},
    {"id": 21, "category": "sequential", "task": "First find the capital of India, then find its population.", "expected": "32"},
    {"id": 22, "category": "sequential", "task": "Calculate 50 times 4, then add 100 to the result.", "expected": "300"},
    {"id": 23, "category": "sequential", "task": "Find the height of Mount Everest, then divide it by 2.", "expected": "4424"},
    {"id": 24, "category": "sequential", "task": "Calculate 10 squared, then multiply the result by 3.", "expected": "300"},
    {"id": 25, "category": "sequential", "task": "Find the length of the longest river, then subtract 650 from it.", "expected": "6000"},
    {"id": 26, "category": "single_lookup", "task": "What is the capital of Canada?", "expected": "ottawa"},
    {"id": 27, "category": "single_lookup", "task": "Who is the CEO of Apple?", "expected": "cook"},
    {"id": 28, "category": "single_lookup", "task": "What is the largest desert in the world?", "expected": "sahara"},
    {"id": 29, "category": "single_lookup", "task": "What is the population of Sydney?", "expected": "5.3"},
    {"id": 30, "category": "single_lookup", "task": "How many continents are there on Earth?", "expected": "7"},
    {"id": 31, "category": "arithmetic", "task": "What is 640 divided by 16?", "expected": "40"},
    {"id": 32, "category": "arithmetic", "task": "What is 35 percent of 2000?", "expected": "700"},
    {"id": 33, "category": "arithmetic", "task": "What is 18 multiplied by 45?", "expected": "810"},
    {"id": 34, "category": "arithmetic", "task": "What is the square of 31?", "expected": "961"},
    {"id": 35, "category": "arithmetic", "task": "What is 12345 plus 54321?", "expected": "66666"},
    {"id": 36, "category": "multi_step", "task": "What is half the population of Sydney in millions?", "expected": "2.65"},
    {"id": 37, "category": "multi_step", "task": "Find the distance from Earth to the Moon in km and divide it by 1000.", "expected": "384.4"},
    {"id": 38, "category": "multi_step", "task": "What is the population of Cairo plus the population of Sydney, in millions?", "expected": "27.3"},
    {"id": 39, "category": "multi_step", "task": "Find the normal human body temperature in Celsius and multiply it by 10.", "expected": "370"},
    {"id": 40, "category": "multi_step", "task": "Find the number of continents on Earth and multiply it by 25.", "expected": "175"},
    {"id": 41, "category": "tool_selection", "task": "What is 850 minus 350? Verify with a tool.", "expected": "500"},
    {"id": 42, "category": "tool_selection", "task": "What is the capital of Brazil?", "expected": "brasilia"},
    {"id": 43, "category": "tool_selection", "task": "Compute 9 times 111.", "expected": "999"},
    {"id": 44, "category": "tool_selection", "task": "What is the largest ocean?", "expected": "pacific"},
    {"id": 45, "category": "tool_selection", "task": "What is 5 percent of 640?", "expected": "32"},
    {"id": 46, "category": "sequential", "task": "First find the capital of Canada, then find its population.", "expected": "1.4"},
    {"id": 47, "category": "sequential", "task": "Calculate 25 times 8, then divide the result by 4.", "expected": "50"},
    {"id": 48, "category": "sequential", "task": "Find the distance from Earth to the Moon in km, then subtract 4400 from it.", "expected": "380000"},
    {"id": 49, "category": "sequential", "task": "Calculate 15 squared, then add 75 to the result.", "expected": "300"},
    {"id": 50, "category": "sequential", "task": "Find the population of Cairo, then multiply it by 2.", "expected": "44"},
]
print(f"Benchmark A loaded: {len(BENCHMARK_50)} tasks")


# %% ==========================================================================
# CELL 6 — BENCHMARK B: GSM8K (same 100 problems — same seed=42)
# ==============================================================================
from datasets import load_dataset

gsm8k_raw = load_dataset("gsm8k", "main", split="test",
                         trust_remote_code=True)
gsm8k_raw = gsm8k_raw.shuffle(seed=42).select(range(100))

GSM8K_TASKS = []
for i, ex in enumerate(gsm8k_raw):
    gold = float(ex["answer"].split("####")[-1].strip().replace(",", ""))
    GSM8K_TASKS.append({"id": i + 1, "task": ex["question"], "gold": gold})

GSM8K_SYSTEM_PROMPT = """You are a helpful AI agent. You solve math word problems step by step using a calculator tool.

Available tools:
- calculator[expression]: evaluates a math expression, e.g. calculator[3 * (12 + 5)]

Use this exact format:
Thought: <your reasoning>
Action: calculator[<expression>]
Observation: <tool result>
... (repeat Thought/Action/Observation as needed)
Thought: I now know the answer.
Final Answer: <the final number only>

Example:
Task: A shop sells pens at 4 dollars each. Tom buys 3 pens and pays with a 20 dollar bill. How much change does he get?
Thought: First I compute the cost of 3 pens.
Action: calculator[3 * 4]
Observation: 12
Thought: Now I subtract the cost from 20.
Action: calculator[20 - 12]
Observation: 8
Thought: I now know the answer.
Final Answer: 8"""

GSM8K_TOOLS = {"calculator": tool_calculator}

def extract_last_number(text):
    nums = re.findall(r"[-+]?\d[\d,]*\.?\d*", text.replace("$", ""))
    if not nums:
        return None
    try:
        return float(nums[-1].replace(",", "").rstrip("."))
    except ValueError:
        return None

def score_gsm8k(item, answer_text):
    pred = extract_last_number(answer_text)
    return pred is not None and abs(pred - item["gold"]) < 1e-3

print(f"Benchmark B loaded: {len(GSM8K_TASKS)} GSM8K problems")


# %% ==========================================================================
# CELL 7 — BENCHMARK C: HotpotQA (same 100 questions — same seed=42)
# ==============================================================================
hotpot_raw = load_dataset("hotpot_qa", "distractor", split="validation",
                          trust_remote_code=True)
hotpot_raw = hotpot_raw.shuffle(seed=42)

HOTPOT_TASKS = []
for ex in hotpot_raw:
    ans = ex["answer"].strip()
    if ans.lower() in ("yes", "no") or len(ans) == 0:
        continue
    titles = ex["context"]["title"]
    paragraphs = [" ".join(sents) for sents in ex["context"]["sentences"]]
    HOTPOT_TASKS.append({
        "id": len(HOTPOT_TASKS) + 1,
        "task": ex["question"],
        "gold": ans,
        "titles": titles,
        "paragraphs": paragraphs,
    })
    if len(HOTPOT_TASKS) == 100:
        break

HOTPOT_SYSTEM_PROMPT = """You are a helpful AI agent. You answer questions by searching a set of documents. Some questions need TWO searches: first find one fact, then search again using that fact.

Available tools:
- search[query]: returns the most relevant document passage for the query

Use this exact format:
Thought: <your reasoning>
Action: search[<query>]
Observation: <passage>
... (repeat Thought/Action/Observation as needed)
Thought: I now know the answer.
Final Answer: <a short answer, just the name/date/entity>"""

def make_hotpot_search(item):
    def search(query):
        q_words = set(re.findall(r"\w+", query.lower()))
        scored = []
        for title, para in zip(item["titles"], item["paragraphs"]):
            t_words = set(re.findall(r"\w+", (title + " " + para).lower()))
            title_words = set(re.findall(r"\w+", title.lower()))
            score = len(q_words & t_words) + 3 * len(q_words & title_words)
            scored.append((score, title, para))
        scored.sort(key=lambda x: -x[0])
        best = scored[0]
        if best[0] == 0:
            return "No results found."
        return f"[{best[1]}] {best[2][:600]}"
    return search

def normalize_answer(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in string.punctuation)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def score_hotpot(item, answer_text):
    gold = normalize_answer(item["gold"])
    pred = normalize_answer(answer_text)
    return len(gold) > 0 and gold in pred

print(f"Benchmark C loaded: {len(HOTPOT_TASKS)} HotpotQA questions")


# %% ==========================================================================
# CELL 8 — EVALUATION HARNESS (identical, incremental saving)
# ==============================================================================
RESULTS = {}

def evaluate(benchmark, config, items, agent_fn, scorer, save_every=10):
    rows = []
    path = f"/workdir/exp4_{benchmark}_{config}.csv"
    print("=" * 70)
    print(f"RUNNING: {benchmark} | {config} | {len(items)} tasks")
    print("=" * 70)
    for i, item in enumerate(items, 1):
        try:
            ans, steps, calls, lat, trace = agent_fn(item)
        except Exception as e:
            ans, steps, calls, lat = f"AGENT ERROR: {e}", 0, 0, 0.0
        correct = scorer(item, ans)
        rows.append({
            "id": item["id"],
            "category": item.get("category", benchmark),
            "task": item["task"][:120],
            "gold": str(item.get("expected", item.get("gold")))[:60],
            "answer": str(ans)[:120],
            "correct": correct, "steps": steps,
            "tool_calls": calls, "latency_s": round(lat, 2),
        })
        mark = "✅" if correct else "❌"
        print(f"{mark} [{i:03d}/{len(items)}] {lat:5.1f}s  {str(ans)[:55]}")
        if i % save_every == 0 or i == len(items):
            pd.DataFrame(rows).to_csv(path, index=False)
    df = pd.DataFrame(rows)
    RESULTS[(benchmark, config)] = df
    acc = 100 * df["correct"].mean()
    print(f"\n>>> {benchmark} | {config}: {acc:.1f}% "
          f"({df['correct'].sum()}/{len(df)}), "
          f"avg latency {df['latency_s'].mean():.2f}s\n")
    return df

def agent_custom(item):
    return run_agent(item["task"], CUSTOM_TOOLS, CUSTOM_SYSTEM_PROMPT,
                     max_steps=6, max_new_tokens=200)

def agent_gsm8k(item):
    return run_agent(item["task"], GSM8K_TOOLS, GSM8K_SYSTEM_PROMPT,
                     max_steps=8, max_new_tokens=260)

def agent_hotpot(item):
    tools = {"search": make_hotpot_search(item)}
    return run_agent(item["task"], tools, HOTPOT_SYSTEM_PROMPT,
                     max_steps=6, max_new_tokens=220)

def score_custom(item, ans):
    return item["expected"].lower() in ans.lower()

print("Harness ready.")


# %% ==========================================================================
# CELL 9 — EVAL 1/3: FP16 on Custom-50   (~10-15 min)
# ==============================================================================
evaluate("custom50", "fp16", BENCHMARK_50, agent_custom, score_custom)


# %% ==========================================================================
# CELL 10 — EVAL 2/3: FP16 on GSM8K   (~30-60 min)
# ==============================================================================
evaluate("gsm8k", "fp16", GSM8K_TASKS, agent_gsm8k, score_gsm8k)


# %% ==========================================================================
# CELL 11 — EVAL 3/3: FP16 on HotpotQA   (~25-45 min)
# ==============================================================================
evaluate("hotpotqa", "fp16", HOTPOT_TASKS, agent_hotpot, score_hotpot)


# %% ==========================================================================
# CELL 12 — FINAL TABLES 4 AND 5 (all three configs, unified harness)
# INT4 and INT4+QLoRA numbers below are YOUR measured Experiment 4 results.
# If your numbers differ (e.g. you re-ran), edit them here.
# ==============================================================================

fp16_c50 = RESULTS[("custom50", "fp16")]
fp16_row = {"Config": "FP16",
            "Overall %": round(100 * fp16_c50["correct"].mean(), 1),
            "Avg Latency (s)": round(fp16_c50["latency_s"].mean(), 2)}
for cat in ["single_lookup", "arithmetic", "multi_step",
            "tool_selection", "sequential"]:
    sub = fp16_c50[fp16_c50["category"] == cat]
    fp16_row[cat + " %"] = round(100 * sub["correct"].mean(), 1)

table4_final = pd.DataFrame([
    fp16_row,
    {"Config": "INT4", "Overall %": 98.0, "Avg Latency (s)": 6.96,
     "single_lookup %": 100.0, "arithmetic %": 100.0, "multi_step %": 90.0,
     "tool_selection %": 100.0, "sequential %": 100.0},
    {"Config": "INT4 + QLoRA", "Overall %": 98.0, "Avg Latency (s)": 7.37,
     "single_lookup %": 100.0, "arithmetic %": 100.0, "multi_step %": 90.0,
     "tool_selection %": 100.0, "sequential %": 100.0},
])
print("\nTABLE 4 (FINAL) — Custom 50-task agentic benchmark, unified harness")
print(table4_final.to_string(index=False))
table4_final.to_csv("/workdir/table4_final.csv", index=False)

fp16_gsm = RESULTS[("gsm8k", "fp16")]
fp16_hot = RESULTS[("hotpotqa", "fp16")]

table5_final = pd.DataFrame([
    {"Benchmark": "GSM8K (100)", "Config": "FP16",
     "Accuracy %": round(100 * fp16_gsm["correct"].mean(), 1),
     "Avg Latency (s)": round(fp16_gsm["latency_s"].mean(), 2),
     "Avg Tool Calls": round(fp16_gsm["tool_calls"].mean(), 2)},
    {"Benchmark": "GSM8K (100)", "Config": "INT4",
     "Accuracy %": 68.0, "Avg Latency (s)": 12.31, "Avg Tool Calls": 3.22},
    {"Benchmark": "GSM8K (100)", "Config": "INT4 + QLoRA",
     "Accuracy %": 65.0, "Avg Latency (s)": 14.31, "Avg Tool Calls": 3.25},
    {"Benchmark": "HotpotQA (100)", "Config": "FP16",
     "Accuracy %": round(100 * fp16_hot["correct"].mean(), 1),
     "Avg Latency (s)": round(fp16_hot["latency_s"].mean(), 2),
     "Avg Tool Calls": round(fp16_hot["tool_calls"].mean(), 2)},
    {"Benchmark": "HotpotQA (100)", "Config": "INT4",
     "Accuracy %": 44.0, "Avg Latency (s)": 9.03, "Avg Tool Calls": 1.47},
    {"Benchmark": "HotpotQA (100)", "Config": "INT4 + QLoRA",
     "Accuracy %": 38.0, "Avg Latency (s)": 7.30, "Avg Tool Calls": 1.13},
])
print("\nTABLE 5 (FINAL) — Standard benchmarks, unified harness")
print(table5_final.to_string(index=False))
table5_final.to_csv("/workdir/table5_final.csv", index=False)

# GPU memory comparison for the paper (FP16 vs INT4 footprint)
print(f"\nFP16 memory footprint: {model.get_memory_footprint()/1e9:.2f} GB "
      f"(INT4 was ~2.3 GB — report both in the paper)")

try:
    import wandb
    run = wandb.init(project="slm-agentic-ai", name="exp4b-fp16-unified")
    run.log({"table4_final": wandb.Table(dataframe=table4_final),
             "table5_final": wandb.Table(dataframe=table5_final)})
    run.finish()
    print("Logged to W&B ✅")
except Exception as e:
    print(f"W&B logging skipped: {e}")


# %% ==========================================================================
# CELL 13 — ZIP RESULTS FOR DOWNLOAD (do this — cloud notebook working dir resets!)
# ==============================================================================
"""
!zip -r /workdir/exp4b_fp16_results.zip /workdir/exp4_*_fp16.csv /workdir/table4_final.csv /workdir/table5_final.csv
!ls -lh /workdir/
"""

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

...

wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: anon-user (anon-entity) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Logged to W&B ✅


'\n!zip -r /workdir/exp4b_fp16_results.zip /workdir/exp4_*_fp16.csv /workdir/table4_final.csv /workdir/table5_final.csv\n!ls -lh /workdir/\n'

In [ ]:
!pip install -q -U "transformers==4.41.2" "accelerate==0.30.1"
!pip install -q "datasets==2.19.1" wandb
print("Install complete. NOW RESTART THE KERNEL before continuing.")

In [2]:
import transformers, torch, datasets
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("datasets:", datasets.__version__)
print("cuda:", torch.version.cuda)
print("gpu:", torch.cuda.get_device_name(0))

transformers: 4.41.2
torch: 2.10.0+cu128
datasets: 2.19.1
cuda: 12.8
gpu: Tesla T4
